# Модуль 06 · Урок 2. Память, оптимизация типов и безопасное downcast

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Как посчитать расход памяти
`df.memory_usage(deep=True)` выдаёт оценку памяти по столбцам, включая объекты и строки. Для общей суммы нужен `.sum()`. Числовой тип `float64` занимает обычно 8 байт на значение; `int16` — 2 байта. У строк сложнее: расход зависит от хранения текста и повторов. Размер CSV на диске не равен занимаемой RAM: текстовый файл при чтении становится структурами в памяти.

In [ ]:
import pandas as pd
values = pd.DataFrame({'count':[1,2,3], 'tag':['x','x','y']})
print(values.memory_usage(deep=True))

## 2. Сжатие числовых типов
`pd.to_numeric(...,downcast='integer')` выбирает более узкий целочисленный тип, если значения допускают это. Downcast может снизить расход RAM, но нельзя терять точность и диапазон: число 100000 не помещается в int16. Для вещественных `float32` экономит память ценой точности; для финансовых расчётов это может быть недопустимо. **Сначала проверяйте допустимую точность, затем изменяйте тип.**

In [ ]:
small = pd.Series([0,12,100])
print(pd.to_numeric(small,downcast='integer').dtype)

## 3. Категории и строки
В колонке с несколькими повторяющимися статусами тип `category` обычно хранит компактный словарь значений и коды; при почти полностью уникальном тексте выигрыш может отсутствовать. В pandas 3.x string dtype отличается от старого object и может использовать PyArrow. Помните, что `astype('category')` следует применять к *категориям*, а не к свободному тексту или ID высокой кардинальности.

## 4. Избежание копий и лишних колонок
В pandas 3.x работает Copy-on-Write: промежуточная выборка логически независима при изменениях. Но это не отменяет расходов памяти на новые столбцы и операции вроде конкатенации десятков гигантских фреймов. Старайтесь хранить только необходимые столбцы, заранее фильтровать данные при чтении и удалять неиспользуемые объекты в долгих пайплайнах.

**Проверьте себя:** чем память таблицы отличается от размера CSV? Когда не надо downcast float64 до float32? Почему категориальный тип выгоден не всегда?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m6_06 · memory bytes

Получите общее число байт DataFrame по `memory_usage(deep=True).sum()`, включая индекс.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_06: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_06 = expected_for("m6_06", df=df)


In [ ]:
# m6_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Результат — числовая оценка размера памяти, зависящая от версии pandas и представления строк.

`int`. Результат для показанного примера: `2830`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_06", result, _expected_m6_06)

### m6_07 · downcast qty

Получите копию таблицы, в которой qty безопасно сжато через `pd.to_numeric(...,downcast="integer")`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_07: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_07 = expected_for("m6_07", df=df)


In [ ]:
# m6_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_07", result, _expected_m6_07)

### m6_08 · categorize region

Получите копию, где city и category приведены к dtype category, не меняя остальные колонки.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_08: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_08 = expected_for("m6_08", df=df)


In [ ]:
# m6_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_08", result, _expected_m6_08)

### m6_09 · drop heavy columns

Получите DataFrame только с колонками keep, в указанном порядке. Не удаляйте их из исходника.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `keep` — список названий столбцов; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `keep`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_09: используются переменные df, keep

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

keep = ['order_id', 'status', 'qty']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_09 = expected_for("m6_09", df=df, keep=keep)


In [ ]:
# m6_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, keep
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 3 столбца; колонки: `order_id`, `status`, `qty`.

```text
  order_id     status  qty
z  demo_o1       paid    2
b  demo_o2        new    1
a  demo_o3       paid    4
d  demo_o4       paid    1
x  demo_o5  cancelled    3
q  demo_o6        new    0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_09", result, _expected_m6_09)

### m6_10 · column bytes

Получите Series с расходом памяти **каждого столбца** в байтах без индекса; порядок столбцов сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_10: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_10 = expected_for("m6_10", df=df)


In [ ]:
# m6_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Результат — числовая оценка размера памяти, зависящая от версии pandas и представления строк.

`pd.Series`: 10 значений; имя серии: `None`.

```text
order_id     336
client_id     48
city         464
category     440
price         48
qty           48
... (показаны только первые значения)
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_10", result, _expected_m6_10)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()